# Solutions for Week 12 Activity: Text as Data and LLMs

# Setup


In [1]:
## imports
import pandas as pd
import numpy as np
import re

## nltk imports
import nltk
from nltk.tokenize import word_tokenize, wordpunct_tokenize
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
_ = nltk.download(['punkt', 'punkt_tab', 'stopwords'], quiet = True)

## spacy for named entity recognition --- if you get an error,
## run this in terminal: python -m spacy download en_core_web_sm
import spacy
nlp = spacy.load('en_core_web_sm')

## sentiment analysis
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

## calling LLM APIs
import requests
import yaml
import time

## sklearn imports
from sklearn.feature_extraction.text import CountVectorizer

## lda
from gensim import corpora
import gensim

## visualization - if you have issues installing, comment out
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis

## repeated printouts
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

/Users/rj545/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/rj545/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


# Setup code for the Airbnb listings 

In [2]:
## load data (path relative to the activities folder)
ab = pd.read_csv("../datasets/airbnb_text.zip")
ab.head()

,id,name,name_upper,neighbourhood_group,price
0,2539,Clean & quiet apt home by the park,CLEAN & QUIET APT HOME BY THE PARK,Brooklyn,149
1,2595,Skylit Midtown Castle,SKYLIT MIDTOWN CASTLE,Manhattan,225
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,THE VILLAGE OF HARLEM....NEW YORK !,Manhattan,150
3,3831,Cozy Entire Floor of Brownstone,COZY ENTIRE FLOOR OF BROWNSTONE,Brooklyn,89
4,5022,Entire Apt: Spacious Studio/Loft by central park,ENTIRE APT: SPACIOUS STUDIO/LOFT BY CENTRAL PARK,Manhattan,80


In [3]:
## stopword list from the slides: nltk english stopwords + custom words
list_stopwords = stopwords.words("english")
custom_words_toadd = ['apartment', 'new york', 'nyc',
                      'bronx', 'brooklyn',
                      'manhattan', 'queens',
                      'staten island']
list_stopwords_new = list_stopwords + custom_words_toadd

## lowercase corpus
corpus_lower = ab.name.str.lower().to_list()

## initialize stemmer
porter = PorterStemmer()

In [4]:
## document-term matrix function from the slides
def create_dtm(list_of_strings, metadata):
    vectorizer = CountVectorizer(lowercase = True)
    dtm_sparse = vectorizer.fit_transform(list_of_strings)
    dtm_dense_named = pd.DataFrame(dtm_sparse.todense(),
                                   columns = vectorizer.get_feature_names_out())
    metadata = metadata.copy()
    metadata.columns = ["metadata_" + col for col in metadata.columns]
    dtm_dense_named_withid = pd.concat([metadata.reset_index(),
                                        dtm_dense_named], axis = 1)
    return dtm_dense_named_withid

In [5]:
## random sample of 1000 listings from the slides
ab_small = (ab.loc[~ab.name.isnull(),
                   ['id', 'neighbourhood_group', 'price', 'name']]
            .copy()
            .rename(columns = {'price': 'price_rawdata'})
            .sample(n = 1000, random_state = 9899))
ab_small['name_lower'] = ab_small['name'].str.lower()
ab_small.head()

,id,neighbourhood_group,price_rawdata,name,name_lower
22540,18227529,Manhattan,140,Live in New York Near Central Park and Columbi...,live in new york near central park and columbi...
47531,35794273,Brooklyn,130,Hope Garden,hope garden
33906,26858196,Brooklyn,75,AWESOME 2 BEDS - QUEEN + SOFA - NEXT TO METRO,awesome 2 beds - queen + sofa - next to metro
12047,9369514,Brooklyn,25,STARTUP CHEAP PLACE BROOKLYN,startup cheap place brooklyn
2908,1669149,Manhattan,250,Beautiful Modern Midtown Apartment,beautiful modern midtown apartment


# Activity 1: named entity recognition

Play around with different variations of the tweet below and look at the results. E.g.:

- What happens if you remove "in the year 2000"?
- What if you make AOL lowercase?
- What if you add Kara Swisher's name?
- What if you add that she's a journalist?

In [6]:
## tweet from the slides (modified from a real tweet)
m_tweet = """When AOL was founded in the year 2000 I remember raising my hand
and asking 'well...if you're using my data - where is my $50'
and no one had a good answer...and there still isn't one.
"""

In [7]:
## helper function to print the named entities in a string
def print_ents(one_string):
    for one_tok in nlp(one_string).ents:
        print("Entity: " + one_tok.text + "; NER tag: " + one_tok.label_)

## original tweet
print_ents(m_tweet)

Entity: AOL; NER tag: ORG
Entity: the year 2000; NER tag: DATE
Entity: 50; NER tag: MONEY


In [8]:
## A. remove "in the year 2000"
m_tweet_noyear = m_tweet.replace(" in the year 2000", "")
print_ents(m_tweet_noyear)

Entity: AOL; NER tag: ORG
Entity: 50; NER tag: MONEY


In [9]:
## B. make AOL lowercase
m_tweet_lower = m_tweet.replace("AOL", "aol")
print_ents(m_tweet_lower)

Entity: the year 2000; NER tag: DATE
Entity: 50; NER tag: MONEY


In [10]:
## C. add Kara Swisher's name
m_tweet_name = m_tweet.replace("I remember raising my hand",
                               "I remember raising my hand to Kara Swisher")
print_ents(m_tweet_name)

Entity: AOL; NER tag: ORG
Entity: the year 2000; NER tag: DATE
Entity: Kara Swisher; NER tag: PERSON
Entity: 50; NER tag: MONEY


In [11]:
## D. add that she's a journalist
m_tweet_journalist = m_tweet.replace("I remember raising my hand",
                                     "I remember raising my hand to the journalist Kara Swisher")
print_ents(m_tweet_journalist)

Entity: AOL; NER tag: ORG
Entity: the year 2000; NER tag: DATE
Entity: Kara Swisher; NER tag: PERSON
Entity: 50; NER tag: MONEY


**Takeaways:** entity tags depend heavily on capitalization and context (e.g., lowercase "aol" may no longer be tagged as an `ORG`); names are typically tagged as `PERSON`; and descriptors like "journalist" are not entities themselves but can change how nearby words are tagged.

# Activity 2: preprocess all the listings

In the slides, we performed preprocessing on a single Airbnb listing. We want to generalize this preprocessing across all listings.

- Embed step two (remove stopwords) and step three (stem) into one or two functions that take in a raw string (e.g., the raw text of an Airbnb listing) and return a preprocessed string
- Apply the function to all the texts in `corpus_lower` created in the setup code block above

In [12]:
def process_step1(one_str):
    try:
        ## remove stopwords
        nostop_listing1 = [word for word in wordpunct_tokenize(one_str)
                           if word not in list_stopwords_new]
        ## stem, keeping only alphabetic tokens with 3+ characters
        clean_listing = [porter.stem(word) for word in nostop_listing1
                         if word.isalpha()
                         and len(word) > 2]
        clean_listing_str = " ".join(clean_listing)
        return clean_listing_str
    except:
        ## missing listings (NaN) aren't strings
        return ""

cleaned_listings = [process_step1(one_listing) for one_listing in
                    corpus_lower]
cleaned_listings[0:5]

['clean quiet apt home park',
 'skylit midtown castl',
 'villag harlem new york',
 'cozi entir floor brownston',
 'entir apt spaciou studio loft central park']

In [14]:
ab['proc_name'] = cleaned_listings
ab[['name', 'proc_name']].head()

,name,proc_name
0,Clean & quiet apt home by the park,clean quiet apt home park
1,Skylit Midtown Castle,skylit midtown castl
2,THE VILLAGE OF HARLEM....NEW YORK !,villag harlem new york
3,Cozy Entire Floor of Brownstone,cozi entir floor brownston
4,Entire Apt: Spacious Studio/Loft by central park,entir apt spaciou studio loft central park


# Activity 3: document-term matrix with the preprocessed text

- Stick with the same random sample of 1000 (`ab_small`)
- Apply the preprocessing steps from Activity 2 to create a new column in `ab_small` with the preprocessed text (if you got stuck on that, try just removing stopwords)
- Use the `create_dtm` function to create a document-term matrix from the preprocessed data
- Take the sum of each of the term columns to find the top words

In [15]:
ab_small['processed'] = [process_step1(one_list)
                         for one_list in ab_small.name_lower.to_list()]

In [16]:
metadata_list = ['id', 'neighbourhood_group', 'price_rawdata']
dtm_pre = create_dtm(list_of_strings = ab_small.processed,
                     metadata = ab_small[metadata_list])

dtm_pre_termsonly = dtm_pre[[col for col in dtm_pre.columns if
                             "metadata" not in col and col != "index"]]

dtm_pre_termsonly.sum().sort_values(ascending = False).head(20)

room            208
bedroom         167
privat          146
cozi            134
apt              97
studio           94
spaciou          83
park             79
sunni            66
east             64
near             56
beauti           52
bed              50
central          47
williamsburg     47
home             44
villag           44
bright           42
luxuri           41
heart            40
dtype: int64

# Activity 4: topic model with the preprocessed text

- Repeat the steps for estimating the topic model, this time with the preprocessed texts
- Play around with other parameters like the number of topics and the filtering bounds --- what seems to produce useful topics?

In [17]:
## drop listings that are empty after preprocessing
ab_small = ab_small[ab_small.processed != ""].copy()

## tokenize the preprocessed texts
tokenized_text = [wordpunct_tokenize(one_text)
                  for one_text in ab_small.processed]

In [18]:
### create dictionary
text_proc_dict = corpora.Dictionary(tokenized_text)

### filter dictionary: word must appear in at least 2% of docs
### (no_below is a count) and no more than 98% (no_above is a fraction)
text_proc_dict.filter_extremes(no_below = round(ab_small.shape[0]*0.02),
                               no_above = 0.98)

### create corpus from dictionary
corpus_fromdict_proc = [text_proc_dict.doc2bow(one_text)
                        for one_text in tokenized_text]

In [19]:
### estimate model
n_topics = 3
ldamod_proc = gensim.models.ldamodel.LdaModel(corpus_fromdict_proc,
                                              num_topics = n_topics,
                                              id2word = text_proc_dict,
                                              passes = 6, alpha = 'auto',
                                              per_word_topics = True,
                                              random_state = 91988)

### print topics and words
topics = ldamod_proc.print_topics(num_words = 15)
for topic in topics:
    print(topic)

(0, '0.264*"room" + 0.111*"privat" + 0.076*"spaciou" + 0.054*"bright" + 0.052*"luxuri" + 0.045*"loft" + 0.044*"sunni" + 0.038*"larg" + 0.037*"min" + 0.030*"bed" + 0.024*"quiet" + 0.023*"close" + 0.020*"great" + 0.019*"subway" + 0.018*"williamsburg"')
(1, '0.181*"bedroom" + 0.105*"apt" + 0.087*"park" + 0.058*"near" + 0.050*"central" + 0.045*"home" + 0.042*"beauti" + 0.037*"charm" + 0.033*"one" + 0.032*"spaciou" + 0.029*"view" + 0.028*"new" + 0.024*"sunni" + 0.022*"bushwick" + 0.018*"locat"')
(2, '0.136*"cozi" + 0.105*"studio" + 0.073*"east" + 0.065*"privat" + 0.043*"villag" + 0.041*"williamsburg" + 0.038*"heart" + 0.032*"side" + 0.032*"upper" + 0.032*"modern" + 0.027*"midtown" + 0.026*"west" + 0.025*"harlem" + 0.024*"garden" + 0.023*"prime"')


In [20]:
### visualize
pyLDAvis.enable_notebook()
lda_display_proc = gensimvis.prepare(ldamod_proc, corpus_fromdict_proc, text_proc_dict)
pyLDAvis.display(lda_display_proc)

/Users/rj545/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/rj545/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/rj545/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/rj545/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed)

## Extension: which topics are associated with higher listing prices?

Same approach as the end of the slides, now with the model estimated on preprocessed text.

In [21]:
## long dataframe: one row per document x topic
topic_probs_long = pd.DataFrame(
    [(doc_id, topic, prob)
     for doc_id, one_doc in zip(ab_small.id, corpus_fromdict_proc)
     for topic, prob in ldamod_proc.get_document_topics(one_doc,
                                                        minimum_probability = 0)],
    columns = ['doc_id', 'topic', 'probability'])

## pivot to wide: one row per document, one col per topic
topic_probs_wide = topic_probs_long.pivot(index = 'doc_id', columns = 'topic',
                                          values = 'probability')
topic_probs_wide.columns = ["topic_" + str(i) for i in topic_probs_wide.columns]
topic_probs_wide = topic_probs_wide.reset_index()

## merge with original data and find each listing's top topic
topic_wmeta = pd.merge(topic_probs_wide, ab_small,
                       left_on = 'doc_id', right_on = 'id')
topic_wmeta['toptopic'] = topic_wmeta[[col for col in topic_wmeta.columns
                                       if "topic_" in col]].idxmax(axis = 1)
topic_wmeta.sample(n = 5, random_state = 555)

## mean price by top topic
topic_wmeta.groupby('toptopic').agg({'price_rawdata': 'mean'})

## mean price by top topic and borough
topic_wmeta.groupby(['toptopic',
                     'neighbourhood_group']).agg({'price_rawdata': 'mean'})

,doc_id,topic_0,topic_1,topic_2,id,neighbourhood_group,price_rawdata,name,name_lower,processed,toptopic
24,698882,0.602699,0.200050,0.197251,698882,Brooklyn,69,The Serenity Room in Historic BKLYN,the serenity room in historic bklyn,seren room histor bklyn,topic_0
631,24493582,0.076047,0.555889,0.368063,24493582,Manhattan,135,Charming beautiful studio perfect location!!,charming beautiful studio perfect location!!,charm beauti studio perfect locat,topic_1
553,22018921,0.160931,0.143186,0.695882,22018921,Manhattan,34,Clean and simple Private Ensuite in Manhanttan,clean and simple private ensuite in manhanttan,clean simpl privat ensuit manhanttan,topic_2
200,8472069,0.188014,0.199777,0.612208,8472069,Brooklyn,80,Cute & Comfy Williamsburg Refuge,cute & comfy williamsburg refuge,cute comfi williamsburg refug,topic_2
896,33787778,0.251329,0.230923,0.517748,33787778,Manhattan,165,Uptown 1-bdr apartment. 2 stops to Times Sq!,uptown 1-bdr apartment. 2 stops to times sq!,uptown bdr stop time,topic_2


,price_rawdata
toptopic,
topic_0,144.633588
topic_1,156.627273
topic_2,159.210169


price_rawdata
toptopic neighbourhood_group               
topic_0  Bronx                    70.000000
         Brooklyn                113.892562
         Manhattan               227.752809
         Queens                   75.069767
         Staten Island            55.000000
topic_1  Bronx                    58.000000
         Brooklyn                134.643617
         Manhattan               201.116667
         Queens                  103.338710
         Staten Island           127.166667
topic_2  Bronx                    65.000000
         Brooklyn                128.051020
         Manhattan               185.931429
         Queens                   86.428571

# Activity 5: LLMs for airbnb listing classification

- A. Load your credentials and read through the function `ask_llm` that takes in a prompt and returns the model's text response
- B. Run the code to create a random sample of 20 airbnb listings - `ab_llm` - change the seed to select a different sample
- C. Feed `ask_llm` a prompt to classify a listing as a "budget", "standard", or "luxury" stay. 
- C. Does the mean price differ across the LLM's categories?

## A. Credentials and function

In [29]:
## load creds
with open("../creds.yaml", 'r') as stream:
    try:
        creds = yaml.safe_load(stream)
    except yaml.YAMLError as exc:
        print(exc)

gemini_key = creds['gemini_api']['api_key']

## Gemini's OpenAI-compatible endpoint
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"

## model to use --- see https://ai.google.dev/gemini-api/docs/models
## for current free-tier model names
gemini_model = "gemini-3.5-flash-lite"

In [30]:
def ask_llm(question_text, api_key, model = gemini_model, url = gemini_url,
            max_tries = 5):
    """
    Query an LLM and return a response
    Inputs: 
    question_text: text of a prompt
    api_key: api key for Gemini
    model: gemini model (defaults to 3.5 flash lite)
    url: url for Gemini
    max_tries: how many times to attempt to query in case of error
    Returns:
    content of LLM response 
    """
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }
    ## temperature = 0 makes the output (close to) deterministic
    data = {
        "model": model,
        "temperature": 0,
        ## gemini 3 models "think" before answering; low effort keeps
        ## simple classification tasks fast (thinking can't be turned off)
        "reasoning_effort": "low",
        "messages": [{"role": "user", "content": question_text}]
    }
    for attempt in range(max_tries):
        response = requests.post(url, json = data,
                                 headers = headers, timeout = 120)
        ## success: return the text of the response
        if response.status_code == 200:
            return response.json()['choices'][0]['message']['content'].strip()
        ## model overloaded (503) or rate limit hit (429): wait and retry,
        ## doubling the wait each time (2, 4, 8, ... seconds)
        if response.status_code in [429, 503]:
            wait = 2 ** (attempt + 1)
            print(f"API busy (error {response.status_code}); retrying in {wait} seconds")
            time.sleep(wait)
        else:
            ## other errors (e.g., bad key): stop and show the message
            raise Exception(f"API error {response.status_code}: {response.text}")
    raise Exception(f"API still unavailable after {max_tries} tries; try again later")


## test
ask_llm("In one sentence, what is a large language model?", gemini_key)

'A large language model is an advanced artificial intelligence system trained on vast amounts of text to understand, summarize, generate, and converse in human-like language.'

## B. Randomly sample a subset of listings

In [24]:
ab_llm = (ab.loc[~ab.name.isnull(), ['id', 'name', 'neighbourhood_group', 'price']]
          .sample(n = 20, random_state = 1203)
          .reset_index(drop = True))

API busy (error 503); retrying in 2 seconds


,name,sentiment_llm
0,Cozy 1 Bedroom Apartment in Downtown Flushing,1
1,Super Clean 1 bedroom Upper West Side,1
2,"Designer's picturesque, large flat",1
3,Room w/ Private Bathroom,0
4,Private room with Central Park view,1
5,"St. Marks, East Village penthouse loft w/ patio!",1
6,Cozy Hideaway in Artist’s Industrial Loft,1
7,Central Harlem Private Studio,0
8,"Large Remodeled Home in Red Hook, Brooklyn, NY",0
9,Morden 3 person shared room fully furnished.,0


## C and D. Beyond sentiment: budget, standard, or luxury?

In [31]:
tier_prompt = ("Based on this Airbnb listing title, is the listing advertising a budget, "
               "standard, or luxury stay? Answer with only one word: budget, standard, or luxury. "
               "Here is the listing title: ")

tier_output = []
for one_listing in ab_llm.name:
    try:
        tier_output.append(ask_llm(tier_prompt + one_listing, gemini_key).lower())
    except Exception as e:
        print(e)
        tier_output.append(np.nan)
    time.sleep(4) # stay under the free tier's requests-per-minute limit

ab_llm['tier_llm'] = tier_output
ab_llm[['name', 'price', 'tier_llm']]

## mean price by the LLM's category
ab_llm.groupby('tier_llm').agg(mean_price = ('price', 'mean'),
                               n_listings = ('price', 'size'))

,name,price,tier_llm
0,Cozy 1 Bedroom Apartment in Downtown Flushing,122,standard
1,Super Clean 1 bedroom Upper West Side,130,standard
2,"Designer's picturesque, large flat",103,luxury
3,Room w/ Private Bathroom,65,standard
4,Private room with Central Park view,92,standard
5,"St. Marks, East Village penthouse loft w/ patio!",140,luxury
6,Cozy Hideaway in Artist’s Industrial Loft,195,standard
7,Central Harlem Private Studio,135,standard
8,"Large Remodeled Home in Red Hook, Brooklyn, NY",250,standard
9,Morden 3 person shared room fully furnished.,35,budget


,mean_price,n_listings
tier_llm,,
budget,69.666667,3
luxury,162.400000,5
standard,163.666667,12
